# Ngôn Sắc Việt — Tự động train NLLB Việt ↔ Dân tộc (câu dài)

Notebook này tự động hóa toàn bộ quy trình trên GPU Kaggle:
1. Cài đặt thư viện & lấy mã nguồn `vietlangai-model`.
2. Tải dữ liệu song ngữ đã rà soát (`import_public.py` + `reviewed_pairs.csv` nếu gắn trong `/kaggle/input`).
3. Huấn luyện NLLB-200 cho **Việt ↔ Dân tộc** với upsample câu dài, ghép đa câu, `max_length=256`, `label_smoothing=0.1`.
4. Chấm điểm `chrF++` (tổng & riêng nhóm `16+w`), `BLEU`, tỷ lệ lặp.
5. Đóng gói `/kaggle/working/nllb-ethnic-ready.zip` sẵn sàng nạp vào `ai-model/models/nllb-ethnic/`.


In [ ]:
# === CẤU HÌNH CHẠY TỰ ĐỘNG ===
REPO_URL = "https://github.com/minhduy6868/vietlangai-model.git"
BRANCH = "main"
BASE_MODEL = "facebook/nllb-200-distilled-600M"
TRAIN_MODE = "lora"  # "lora" (nhẹ ~180MB, giữ vi<->en gốc) hoặc "full" (16GB VRAM+)
RANK = 64
EPOCHS = 5
LR = 3e-4
BATCH = 32
GRAD_ACCUM = 4
MAX_LENGTH = 256
LABEL_SMOOTHING = 0.1
CONCAT_RATIO = 0.35
LONG_UPSAMPLE = 2
TEMP = 2.0
INCLUDE_JARAI = False  # Đổi True nếu nhóm đã xác nhận quyền dùng jarai-vie-bible
HF_REPO_ID = ""  # Tùy chọn: "minhduy6868/nllb-ngonsacviet" nếu muốn tự đẩy lên HF Hub


In [ ]:
import shutil
import subprocess
import sys

import torch

smi = shutil.which("nvidia-smi")
if smi:
    print(subprocess.check_output([smi], text=True))
else:
    print("Cảnh báo: không thấy nvidia-smi trong PATH")

if not torch.cuda.is_available():
    raise RuntimeError(
        "Không phát hiện GPU CUDA trên Kaggle! Kiểm tra Accelerator (GPU T4) và Phone Verification."
    )
print("CUDA GPU:", torch.cuda.get_device_name(0))

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.56,<5", "peft>=0.17", "accelerate>=1.0",
    "sacrebleu>=2.4", "sentencepiece>=0.2", "pyarrow>=15", "huggingface_hub",
])


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "vietlangai-model"

if (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)])

# Tự tìm reviewed_pairs.csv hoặc pairs.csv gắn thêm trong /kaggle/input
for fname, dest in [
    ("reviewed_pairs.csv", REPO_DIR / "data" / "reviewed_pairs.csv"),
    ("pairs.csv", REPO_DIR / "data" / "public" / "pairs.csv"),
]:
    found = list(Path("/kaggle/input").rglob(fname))
    if found:
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(found[0], dest)
        print(f"Đã nạp {fname} từ {found[0]}")

os.chdir(REPO_DIR)
print("Working dir:", Path.cwd())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import subprocess
import sys

cmd = [sys.executable, "training/import_public.py"]
if INCLUDE_JARAI:
    cmd += ["--include", "tay-vietnamese-nmt", "rhade-vietnamese-mt", "jarai-vie-bible", "--allow-unknown-license"]
subprocess.check_call(cmd)


In [ ]:
import subprocess
import sys

OUT_DIR = "/kaggle/working/nllb-ethnic"
train_cmd = [
    sys.executable, "training/finetune_nllb.py",
    "--base", BASE_MODEL,
    "--output", OUT_DIR,
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--batch", str(BATCH),
    "--grad-accum", str(GRAD_ACCUM),
    "--max-length", str(MAX_LENGTH),
    "--label-smoothing", str(LABEL_SMOOTHING),
    "--rank", str(RANK),
    "--concat-ratio", str(CONCAT_RATIO),
    "--long-upsample", str(LONG_UPSAMPLE),
    "--temp", str(TEMP),
]
if INCLUDE_JARAI:
    train_cmd.append("--allow-unknown-license")
if TRAIN_MODE == "full":
    train_cmd.append("--full")

print("Chạy:", " ".join(train_cmd))
subprocess.check_call(train_cmd)


In [ ]:
import subprocess
import sys

eval_cmd = [
    sys.executable, "training/evaluate.py",
    "--base", BASE_MODEL,
    "--adapter", "/kaggle/working/nllb-ethnic",
    "--split", "test",
    "--batch", "32",
    "--beams", "4",
    "--output", "/kaggle/working/eval_report.json",
]
if INCLUDE_JARAI:
    eval_cmd.append("--allow-unknown-license")
subprocess.check_call(eval_cmd)


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/vietlangai-model")
from app.nllb import NllbEngine

engine = NllbEngine(BASE_MODEL, "/kaggle/working/nllb-ethnic", beams=4, gated=False)
samples = [
    ("vi", "tyz", "Sáng nay trời mát mẻ, gia đình tôi cùng nhau lên rẫy hái măng. Buổi chiều chúng tôi quay về bản nấu cơm và trò chuyện bên bếp lửa."),
    ("vi", "rad", "Hôm nay buôn làng tổ chức lễ hội đầu mùa rất vui vẻ. Mọi người mặc trang phục truyền thống và cùng nhau đánh chiêng đón khách quý."),
]
for src, tgt, text in samples:
    if (src, tgt) in engine.pairs():
        out = engine.translate(text, src, tgt)
        back = engine.translate(out, tgt, src)
        print(f"[{src}->{tgt}] {out}")
        print(f"[{tgt}->{src}] {back}\n")


In [ ]:
import os
import shutil
from pathlib import Path

bundle_dir = Path("/kaggle/working/nllb-ethnic")
ckpt_dir = bundle_dir / "checkpoints"
if ckpt_dir.exists():
    shutil.rmtree(ckpt_dir)

zip_file = shutil.make_archive("/kaggle/working/nllb-ethnic-ready", "zip", root_dir="/kaggle/working", base_dir="nllb-ethnic")
print("Đã đóng gói:", zip_file)

if HF_REPO_ID:
    token = os.getenv("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if token:
        from huggingface_hub import HfApi
        api = HfApi(token=token)
        api.create_repo(HF_REPO_ID, exist_ok=True)
        api.upload_folder(folder_path=str(bundle_dir), repo_id=HF_REPO_ID)
        print("Đã đẩy model lên Hugging Face Hub:", HF_REPO_ID)
